# Gold Layer — Business-Ready Tables

Transform Silver tables into star-schema tables optimized for analytics and ALS model training.

### 1. SparkSession + Load Source Tables

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, year as spark_year, count, avg, stddev,
    min as spark_min, max as spark_max,
    countDistinct, explode, when, lit,
    round as spark_round, sum as spark_sum,
)
from pyspark.sql.types import FloatType

spark = SparkSession.builder \
    .appName("Gold") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

silver_ratings           = spark.read.parquet("silver/ratings")
silver_movies            = spark.read.parquet("silver/movies")
silver_tags              = spark.read.parquet("silver/tags")
silver_movies_with_links = spark.read.parquet("silver/movies_with_links")
bronze_enrichment        = spark.read.parquet("bronze/enrichment/parquet")

print("=== Source Table Row Counts ===")
for name, df in [("silver/ratings", silver_ratings),
                  ("silver/movies", silver_movies),
                  ("silver/tags", silver_tags),
                  ("silver/movies_with_links", silver_movies_with_links),
                  ("bronze/enrichment", bronze_enrichment)]:
    print(f"  {name:<28} {df.count():>12,}")

### 2. Fact Ratings

In [ ]:
fact_ratings = (
    silver_ratings
    .select("userId", "movieId", "rating", "rated_at")
    .withColumn("rating_year", spark_year("rated_at"))
)

# Assert zero nulls in key columns
for c in ["userId", "movieId", "rating"]:
    null_count = fact_ratings.filter(col(c).isNull()).count()
    assert null_count == 0, f"Unexpected nulls in fact_ratings.{c}: {null_count}"
    print(f"  {c}: 0 nulls")

print(f"\nfact_ratings rows: {fact_ratings.count():,}")
fact_ratings.printSchema()

In [ ]:
fact_ratings.write.mode("overwrite").partitionBy("rating_year").parquet("gold/fact_ratings")
print("gold/fact_ratings written.")

### 3. Dim Users

In [ ]:
# Rating aggregates per user
user_rating_aggs = (
    silver_ratings
    .groupBy("userId")
    .agg(
        count("*").alias("rating_count"),
        spark_round(avg("rating"), 2).alias("avg_rating"),
        spark_round(stddev("rating"), 2).alias("rating_stddev"),
        spark_min("rating").alias("min_rating"),
        spark_max("rating").alias("max_rating"),
        countDistinct("movieId").alias("distinct_movies_rated"),
        spark_min("rated_at").alias("first_rating_at"),
        spark_max("rated_at").alias("last_rating_at"),
    )
)

print(f"user_rating_aggs: {user_rating_aggs.count():,} users")
user_rating_aggs.show(3)

In [ ]:
# Genre breadth per user
user_genre_breadth = (
    silver_ratings.select("userId", "movieId")
    .join(silver_movies.select("movieId", "genres"), "movieId")
    .select("userId", explode("genres").alias("genre"))
    .groupBy("userId")
    .agg(countDistinct("genre").alias("distinct_genres_rated"))
)

# Tag counts per user
user_tag_counts = (
    silver_tags
    .groupBy("userId")
    .agg(count("*").alias("tag_count"))
)

# Assemble dim_users
dim_users = (
    user_rating_aggs
    .join(user_genre_breadth, "userId", "left")
    .join(user_tag_counts, "userId", "left")
    .fillna(0, subset=["distinct_genres_rated", "tag_count"])
    .withColumn(
        "active_years",
        spark_year("last_rating_at") - spark_year("first_rating_at") + 1
    )
    .withColumn(
        "is_power_user",
        (col("rating_count") >= 500)
        & (col("distinct_genres_rated") >= 5)
        & (col("tag_count") >= 1)
    )
)

print(f"dim_users: {dim_users.count():,} rows")
power_users = dim_users.filter(col("is_power_user")).count()
print(f"Power users: {power_users:,}")
dim_users.printSchema()
dim_users.show(5)

In [ ]:
dim_users.write.mode("overwrite").parquet("gold/dim_users")
print("gold/dim_users written.")

### 4. Dim Movies Enriched

In [ ]:
# Pre-aggregate ratings per movie
movie_rating_aggs = (
    silver_ratings
    .groupBy("movieId")
    .agg(
        spark_round(avg("rating"), 2).alias("avg_rating"),
        count("*").alias("rating_count"),
    )
)

# Pre-aggregate tags per movie
movie_tag_counts = (
    silver_tags
    .groupBy("movieId")
    .agg(count("*").alias("tag_count"))
)

In [ ]:
# Clean enrichment: drop metadata columns, convert 0 budget/revenue to null
enrichment_clean = (
    bronze_enrichment
    .drop("_ingestion_timestamp", "_source_file", "title", "tmdbId")
    .withColumn("budget", when(col("budget") == 0, lit(None)).otherwise(col("budget")))
    .withColumn("revenue", when(col("revenue") == 0, lit(None)).otherwise(col("revenue")))
)

print(f"enrichment_clean: {enrichment_clean.count()} rows")
enrichment_clean.printSchema()

In [ ]:
# Assemble dim_movies_enriched
dim_movies_enriched = (
    silver_movies_with_links
    .join(movie_rating_aggs, "movieId", "left")
    .join(movie_tag_counts, "movieId", "left")
    .join(enrichment_clean, "movieId", "left")
    .fillna(0, subset=["rating_count", "tag_count"])
    .withColumn(
        "profit",
        when(col("revenue").isNotNull() & col("budget").isNotNull(),
             col("revenue") - col("budget"))
    )
    .withColumn("has_enrichment", col("directors").isNotNull())
)

print(f"dim_movies_enriched: {dim_movies_enriched.count():,} rows")
enriched_count = dim_movies_enriched.filter(col("has_enrichment")).count()
print(f"Movies with TMDB enrichment: {enriched_count}")
dim_movies_enriched.printSchema()
dim_movies_enriched.show(5, truncate=40)

In [ ]:
dim_movies_enriched.write.mode("overwrite").parquet("gold/dim_movies_enriched")
print("gold/dim_movies_enriched written.")

### 5. Null Audit

In [ ]:
gold_tables = {
    "fact_ratings": spark.read.parquet("gold/fact_ratings"),
    "dim_users": spark.read.parquet("gold/dim_users"),
    "dim_movies_enriched": spark.read.parquet("gold/dim_movies_enriched"),
}

enrichment_cols = {
    "directors", "budget", "revenue", "runtime", "release_date",
    "poster_url", "overview", "vote_average", "original_language", "profit",
}

for name, df in gold_tables.items():
    print(f"\n=== {name} ({df.count():,} rows) ===")
    null_counts = df.select(
        [spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]
    ).collect()[0]
    for c in df.columns:
        n = null_counts[c]
        if n == 0:
            note = ""
        elif name == "dim_movies_enriched" and c in enrichment_cols:
            note = " <- expected (enrichment covers ~500 movies)"
        elif c == "avg_rating":
            note = " <- unrated movies"
        else:
            note = ""
        print(f"  {c:<25} {n:>10,} nulls{note}")

dme = gold_tables["dim_movies_enriched"]
total = dme.count()
enriched = dme.filter(col("has_enrichment")).count()
print(f"\nEnrichment coverage: {enriched}/{total} ({100 * enriched / total:.1f}%)")

### 6. Highest Rated Director

In [ ]:
director_rankings = (
    dme
    .filter(col("has_enrichment") & col("avg_rating").isNotNull())
    .select(explode("directors").alias("director"), "avg_rating", "title", "rating_count")
    .groupBy("director")
    .agg(
        spark_round(avg("avg_rating"), 2).alias("director_avg_rating"),
        count("*").alias("movie_count"),
        spark_sum("rating_count").alias("total_ratings"),
    )
    .filter(col("movie_count") >= 2)
    .orderBy(col("director_avg_rating").desc())
)

print("=== Highest Rated Directors (2+ movies in top 500) ===")
director_rankings.show(20, truncate=False)

### 7. Additional Verification Queries

In [ ]:
# Most profitable movies
print("=== Most Profitable Movies ===")
dme.filter(col("profit").isNotNull()) \
    .select("title", "budget", "revenue", "profit", "avg_rating") \
    .orderBy(col("profit").desc()) \
    .show(10, truncate=False)

# Power User vs Normal User comparison
du = gold_tables["dim_users"]
print("=== Power Users vs Normal Users ===")
du.groupBy("is_power_user").agg(
    count("*").alias("user_count"),
    spark_round(avg("rating_count"), 1).alias("avg_ratings_per_user"),
    spark_round(avg("avg_rating"), 2).alias("avg_of_avg_rating"),
    spark_round(avg("distinct_genres_rated"), 1).alias("avg_genre_breadth"),
    spark_round(avg("tag_count"), 1).alias("avg_tags_per_user"),
).show()

# Partition distribution in fact_ratings
print("=== fact_ratings Partition Distribution ===")
fr = gold_tables["fact_ratings"]
fr.groupBy("rating_year").agg(count("*").alias("rows")) \
    .orderBy("rating_year").show(30)

### 8. ALS Readiness Check

In [ ]:
als_data = fr.select(
    col("userId"),
    col("movieId"),
    col("rating").cast(FloatType()),
)

train, test = als_data.randomSplit([0.8, 0.2], seed=42)
train.cache()
test.cache()

n_users = als_data.select("userId").distinct().count()
n_movies = als_data.select("movieId").distinct().count()
n_ratings = als_data.count()
sparsity = 1 - (n_ratings / (n_users * n_movies))

print("=== ALS Training Readiness ===")
print(f"  Users:    {n_users:>12,}")
print(f"  Movies:   {n_movies:>12,}")
print(f"  Ratings:  {n_ratings:>12,}")
print(f"  Sparsity: {sparsity:>12.4%}")
print(f"  Train:    {train.count():>12,}")
print(f"  Test:     {test.count():>12,}")

train.unpersist()
test.unpersist()
print("\nGold layer complete. Ready for Phase 4 (ALS model training).")